# Statevectors, gates, and global phase

Exercise common one- and two-qubit gates, Qiskit's little-endian ordering, and global phase.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
circuit = QuantumCircuit(3, name="gate-parity")
circuit.h(0)
circuit.ry(0.37, 1)
circuit.cx(0, 2)
circuit.cp(-0.23, 2, 1)
circuit.rxx(0.41, 0, 1)
circuit.ryy(-0.19, 1, 2)
circuit.rzz(0.29, 2, 0)
circuit.global_phase = 0.17

reference, reference_ms, _ = benchmark(
    lambda: np.asarray(Statevector.from_instruction(circuit).data)
)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    return np.asarray(backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"])

candidate, mettleq_ms, _ = benchmark(run_mettleq)
error = phase_aligned_statevector_error(reference, candidate)
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/02_statevectors_and_gates.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-aligned statevector atol=2e-6",
    passed=error <= 2e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": error, "norm": float(np.linalg.norm(candidate))},
)